# Article eligibility and user profiles

Run notebooks 3_01 and 3_02 first.
**Input:** supplied histories, article vectors and chronological impressions/candidates.
**Output:** observation-time eligibility, static supplied-history profiles and causal profile checks.
Static profiles never include logged feedback. In replay, score every request at timestamp t before consuming its labels.
`long_text` is the supplied-history baseline; `text`/`entity` and category interests pool the full causal history; sequence references alone are capped.

Run the cells in order. Code must be available in this runtime, including `src/newsrec`.
For Colab, upload the full checkout or mount its folder on Drive; opening only the notebook does not upload the package.
Set `NEWSREC_CODE` before the setup cell if the checkout is elsewhere. Set `DATA_ROOT` for separate datasets.
Inputs are prepared parquet files under `1_Data_Preparation/processed`; outputs go to `3_Representations/artifacts`.
The calculation cells below are passed to shared storage helpers.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Code: run from the Group 6 checkout, or set NEWSREC_CODE to that folder.
CODE_ROOT = (
    Path(os.environ["NEWSREC_CODE"]).expanduser().resolve()
    if os.environ.get("NEWSREC_CODE")
    else next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/newsrec").is_dir()), None)
)
if CODE_ROOT is None or not (CODE_ROOT / "src/newsrec").is_dir():
    raise FileNotFoundError("Open the project checkout, or set NEWSREC_CODE to the folder containing src/newsrec/.")
sys.path.insert(0, str(CODE_ROOT / 'src'))

# Data: use the checkout by default. For Drive, mount it first and set DATA_ROOT.
DATA_ROOT = None  # Colab example: '/content/drive/MyDrive/BT4222'
ROOT = Path(DATA_ROOT or os.getenv('NEWSREC_ROOT') or CODE_ROOT).expanduser().resolve()
ARTIFACTS = ROOT / '3_Representations/artifacts'
print('Code:', CODE_ROOT)
print('Data and outputs:', ROOT)

from collections import deque

import polars as pl

from newsrec.common import (
    SPLITS,
    Settings,
    normalise,
    saved_stage_ready,
)
from newsrec.pipeline import build_supplied_profiles, profile_preview
from newsrec.representations import build_availability
from newsrec.temporal import HOUR, User
from newsrec.temporal import TemporalState as _StateStorage

Code: /Users/bryan03/Desktop/BT4222-Group-6-
Data and outputs: /Users/bryan03/Desktop/BT4222-Group-6-


In [2]:
SETTINGS = Settings()  # Fixed seed and dimensions; use the same settings in all stages.
REBUILD = False  # Set True after editing a calculation cell. Shared code changes invalidate caches.
# Terminal override applies only to this execution; the saved preference stays False.
REBUILD = REBUILD or os.environ.get("NEWSREC_FORCE_REBUILD") == "1"

## 1. Observation-time eligibility

`available_from` is the earliest observable slate appearance or supplied-history discovery, not publication time.
Do not backdate new test users' histories to training start. A history-only article has no observed exposure age.
Eligibility uses `available_from <= t`; it is separate from counts built from events strictly before `t`.

### Earliest observable article discovery

The minimum observed history/slate timestamp is an eligibility threshold. An article is served only when `available_from <= request_time`; this is not an invented publication date.

In [3]:
def compute_availability(processed):
    histories = []
    exposures = []
    for split in SPLITS:
        histories.append(
            pl.scan_parquet(processed / f"impressions_{split}.parquet").select(
                "user_code", "timestamp", "history"
            )
        )
        exposures.append(
            pl.scan_parquet(processed / f"candidates_{split}.parquet").select(
                "news_code", "timestamp"
            )
        )
    first_hist = (
        pl.concat(histories)
        .group_by("user_code", "history")
        .agg(pl.col("timestamp").min())
        .filter(pl.col("history").is_not_null() & (pl.col("history") != ""))
        .with_columns(pl.col("history").str.split(" ").alias("news_id"))
        # Empty histories were filtered above; the default explosion is sufficient.
        .explode("news_id")
        .with_columns(
            pl.col("news_id").str.strip_prefix("N").cast(pl.Int32).alias("news_code")
        )
        .group_by("news_code")
        .agg(pl.col("timestamp").min().alias("history_discovered_at"))
    )
    first_exp = (
        pl.concat(exposures)
        .group_by("news_code")
        .agg(pl.col("timestamp").min().alias("first_observed_exposure"))
    )
    frame = (
        first_hist.join(first_exp, on="news_code", how="full", coalesce=True)
        .with_columns(
            pl.min_horizontal("history_discovered_at", "first_observed_exposure").alias(
                "available_from"
            )
        )
        .sort("news_code")
        .collect(engine="streaming")
        .to_pandas()
    )
    frame["discovery_source"] = np.where(
        frame.history_discovered_at.isna(),
        "slate",
        np.where(
            frame.first_observed_exposure.isna()
            | (frame.history_discovered_at < frame.first_observed_exposure),
            "history",
            "slate_or_tied",
        ),
    )
    start = min(
        pl.scan_parquet(processed / f"impressions_{s}.parquet")
        .select(pl.col("timestamp").min())
        .collect()
        .item()
        for s in SPLITS
    )
    frame["left_censored"] = frame.first_observed_exposure.eq(start)
    return frame, start

In [4]:
if REBUILD or not saved_stage_ready(ROOT, SETTINGS, "availability"):
    availability = build_availability(
        ROOT, SETTINGS, availability_builder=compute_availability
    )
else:
    print("Loaded saved article availability.")
    availability = pd.read_parquet(ARTIFACTS / "article_availability.parquet")
display(availability.head())
display(availability.discovery_source.value_counts())

,news_code,history_discovered_at,first_observed_exposure,available_from,discovery_source,left_censored
0,1,2019-11-10 12:40:53,NaT,2019-11-10 12:40:53,history,False
1,2,2019-11-09 00:04:31,NaT,2019-11-09 00:04:31,history,False
2,4,2019-11-09 01:25:52,NaT,2019-11-09 01:25:52,history,False
3,5,2019-11-09 08:15:04,NaT,2019-11-09 08:15:04,history,False
4,6,2019-11-09 03:49:30,NaT,2019-11-09 03:49:30,history,False


discovery_source
history          77690
slate            24214
slate_or_tied     2247
Name: count, dtype: int64

## 2. Supplied-history profiles

Pool every valid article in each supplied history; timestamps are unknown, so apply no decay.
Validate all distinct user/history pairs before pooling a user once.
`history_over_cap` means the supplied history exceeds the replay sequence cap; the static pool still uses the full history.
Only `seen_in_train` users may fit user clusters. Validation/test-only profiles are transform inputs at their own request times.

### Supplied-history pooling

Ignore unavailable/zero vectors, mean-pool the valid history vectors and normalise. Supplied histories have no click timestamps, so this static profile adds no invented recency decay.

In [5]:
def normalised_mean(matrix, rows):
    if not len(rows):
        return (np.zeros(matrix.shape[1], dtype=np.float32), 0)
    selected = matrix[np.asarray(rows, dtype=np.int32)]
    valid = np.linalg.norm(selected, axis=1) > 1e-12
    if valid.any():
        pooled = normalise(selected[valid].mean(axis=0))
    else:
        pooled = np.zeros(matrix.shape[1], np.float32)
    return (pooled, int(valid.sum()))


def pool_supplied_history(history, bundle):
    raw = "" if pd.isna(history) else str(history)
    codes = [int(x.removeprefix("N")) for x in raw.split()]
    rows = bundle.rows(codes)
    text, valid_text = normalised_mean(bundle.text, rows) # text embeddings from 3_01 (text64.npy)
    entity, valid_entity = normalised_mean(bundle.entity, rows) # entity embeddings from 3_02 (article_entity100.npy)
    return rows, text, entity, valid_text, valid_entity

In [6]:
if REBUILD or not saved_stage_ready(ROOT, SETTINGS, "profiles"):
    profiles = build_supplied_profiles(
        ROOT, SETTINGS, history_pooler=pool_supplied_history
    )
else:
    print("Loaded saved supplied-history profiles.")
    profiles = pd.read_parquet(ARTIFACTS / "supplied_profile_index.parquet")
display(
    profiles.groupby("seen_in_train").agg(
        users=("user_code", "size"),
        no_history=("history_length", lambda x: int(x.eq(0).sum())),
        no_entity_profile=("valid_entity_history_count", lambda x: int(x.eq(0).sum())),
    )
)
display(profiles.head())

,users,no_history,no_entity_profile
seen_in_train,,,
False,82554,2250,2690
True,667880,11836,15001


,user_id,user_code,profile_row,history_length,valid_text_history_count,valid_entity_history_count,history_over_cap,seen_in_train,has_text_profile,has_entity_profile
0,U0,0,0,8,8,6,False,True,True,True
1,U1,1,1,72,72,57,True,True,True,True
2,U10,2,2,3,3,3,False,True,True,True
3,U100,3,3,43,43,31,False,True,True,True
4,U1000,4,4,9,9,7,False,True,True,True


## 3. Demonstrate causal profile updates

Seed each user once; append genuine earlier click events, retaining repeats. Score all requests in a timestamp
batch before consuming its feedback. Full category counts use all prior clicks; only NRMS/sequence references are capped.
Snapshot chronology is unverified by default, so its capped subset is deterministic and not called recent.
Recent profiles use timestamped in-window clicks only. Up to three category interests are ranked by full
prior category counts and pooled from all valid causal title vectors. Empty groups are masked.

The example processes only three initial timestamp batches, with their preceding state intact.

### Causal updates and full-history interests

`begin`: seed each user once from supplied history.
`profile`: pool text/entity and category-interest vectors from the full causal history; retain capped sequence references and a separate timestamped recent view.
`finish`: apply feedback only after the whole timestamp batch has been scored.

Snapshot click times/order are unknown; use a reproducible random sequence subset without claiming recency.
Repeated earlier clicks remain genuine events. Full-history row references let downstream code reconstruct the same pooled vectors and interests; their memory/storage cost grows with history length.

In [7]:
class TemporalState(_StateStorage):
    """Profile calculations with shared array/SQLite allocation and cleanup."""

    def begin(self, requests, t):
        t = pd.Timestamp(t)
        if self.active_time is not None:
            raise ValueError("Previous timestamp has not been finished")
        if self.last_update is not None and t <= self.last_update:
            raise ValueError(
                "Replay timestamps must increase strictly; merge same-time requests"
            )
        if t < self.start or t > self.end:
            raise ValueError("Request outside state horizon")
        if (
            requests.impression_uid.duplicated().any()
            or not requests.timestamp.eq(t).all()
        ):
            raise ValueError("begin requires unique requests from this timestamp only")
        self.active_time = t
        for row in requests.itertuples(index=False):
            text = "" if pd.isna(row.history) else str(row.history)
            codes = tuple(int(token.removeprefix("N")) for token in text.split())
            key = int(row.user_code)
            if key in self.users:
                if self.users[key].snapshot_key != codes:
                    raise ValueError(f"Supplied history changed for user {key}")
                continue
            snapshot = self.bundle.rows(codes)
            user = User(snapshot, codes, deque(maxlen=max(self.cfg.history_cap, self.cfg.recent_count)))
            user.categories.update(self.cat[snapshot])
            user.subcategories.update(self.subcat[snapshot])
            self.users[key] = user

    def profile(self, user_code, t):
        t = pd.Timestamp(t)
        if self.active_time != t:
            raise ValueError(
                "Call begin() for this timestamp before requesting profiles"
            )
        key = int(user_code)
        user = self.users[key]
        cache_key = (key, user.version, t.value)
        if cache_key in self.profiles:
            return self.profiles[cache_key]
        observed = list(user.observed)
        seed = (
            user.snapshot
            if self.cfg.snapshot_order_verified
            else np.random.default_rng(self.cfg.seed + key).permutation(np.sort(user.snapshot))
        )
        history = np.r_[seed, [a for _, a in observed]].astype(np.int32)[
            -self.cfg.history_cap :
        ]
        recent = [
            a for time, a in observed if t.value - time <= self.cfg.recent_hours * HOUR
        ][-self.cfg.recent_count :]
        # Pool the full causal multiset, preserving repeated clicks. Sequence
        # truncation below must not choose which articles define user interests.
        full_history = np.r_[user.snapshot, user.clicked_rows].astype(np.int32)
        text, valid_text = normalised_mean(self.bundle.text, full_history)
        entity, valid_entity = normalised_mean(self.bundle.entity, full_history)
        long, _ = normalised_mean(self.bundle.text, user.snapshot)
        recent_vector, valid_recent = normalised_mean(self.bundle.text, recent)
        interests = []
        interest_categories = []
        for cat, _ in sorted(user.categories.items(), key=lambda x: (-x[1], x[0]))[
            : self.cfg.interest_count
        ]:
            rows = full_history[self.cat[full_history] == cat]
            vector, valid = normalised_mean(self.bundle.title, rows)
            if valid and np.linalg.norm(vector) > 0:
                interests.append(vector)
                interest_categories.append(cat)
        padded = np.zeros(self.cfg.history_cap, np.int32)
        padded[: len(history)] = history
        result = {
            "text": text,
            "entity": entity,
            "long_text": long,
            "recent_text": recent_vector,
            "interests": np.asarray(interests, np.float32).reshape(
                -1, self.cfg.text_dim
            ),
            "history_rows": padded,
            "full_history_rows": full_history,
            "history_mask": padded != 0,
            "selected_interest_categories": interest_categories,
            "recent_rows": list(map(int, recent)),
            "history_length": len(user.snapshot) + user.n_observed,
            "retained_length": len(history),
            "valid_text_fraction": valid_text / max(1, len(full_history)),
            "valid_entity_fraction": valid_entity / max(1, len(full_history)),
            "has_text": bool(np.linalg.norm(text) > 0),
            "has_entity": bool(np.linalg.norm(entity) > 0),
            "has_recent": bool(valid_recent and np.linalg.norm(recent_vector) > 0),
            "no_supplied_history": len(user.snapshot) == 0,
            "group_count": len(interests),
            "selected_group_count": min(self.cfg.interest_count, len(user.categories)),
            "state_version": user.version,
        }
        self.profiles[cache_key] = result
        return result

    def finish(self, requests, events, t, feedback_splits):
        t = pd.Timestamp(t)
        if self.active_time != t:
            raise ValueError("Timestamp is not active")
        if not events.timestamp.eq(t).all() or not set(events.impression_uid) <= set(
            requests.impression_uid
        ):
            raise ValueError("Feedback must belong to this timestamp's requests")
        owners = requests.set_index("impression_uid").user_code
        if not np.array_equal(
            events.user_code.to_numpy(), events.impression_uid.map(owners).to_numpy()
        ):
            raise ValueError("Feedback/request user mismatch")
        if events.duplicated(["impression_uid", "news_code"]).any():
            raise ValueError("Duplicate feedback event")
        for u in requests.user_code.unique():
            self.users[int(u)].last_impression = t
        selected = events[events.split.isin(feedback_splits)]
        if len(selected):
            if not selected.clicked.isin([0, 1]).all():
                raise ValueError("Feedback labels must be binary")
            rows = self.bundle.rows(selected.news_code)
            labels = selected.clicked.to_numpy(np.int64)
            self.pairs.update(selected.user_code, rows, labels, t)
            np.add.at(self.exposures, rows, 1)
            np.add.at(self.clicks, rows, labels)
            unseen = self.first[rows] < 0
            self.first[rows[unseen]] = t.value
            hour = int(t.value // HOUR - self.start_hour)
            np.add.at(self.hour_exposures[:, hour], rows, 1)
            np.add.at(self.hour_clicks[:, hour], rows, labels)
            self.total_exposures += len(rows)
            self.total_clicks += int(labels.sum())
            clicked = selected.loc[selected.clicked.eq(1)].sort_values(
                ["user_code", "news_code", "impression_uid"]
            )
            for event in clicked.itertuples(index=False):
                user = self.users[int(event.user_code)]
                a = self.bundle.lookup[int(event.news_code)]
                user.observed.append((t.value, a))
                user.clicked_rows.append(a)
                user.n_observed += 1
                user.categories[self.cat[a]] += 1
                user.subcategories[self.subcat[a]] += 1
                user.version += 1
        self.profiles.clear()
        self.last_update = t
        self.active_time = None

In [8]:
from newsrec.audits import check_temporal_profiles

# Output: deterministic regression checks on a tiny, explicit synthetic history.
# Verify dominant interests survive the cap and labels affect only the next timestamp.
display(check_temporal_profiles(TemporalState, SETTINGS))

# Output: an example from real prepared data, not a complete leakage proof.
display(profile_preview(ROOT, SETTINGS, state_factory=TemporalState))
display(pd.read_csv(ROOT / 'results/representation_coverage.csv'))
display(pd.read_csv(ROOT / 'results/representation_cost.csv').tail())

,feedback_label,pre_feedback_clicks,next_timestamp_clicks,interests_beyond_cap,delayed_feedback
0,0,5,5,passed,passed
1,1,5,6,passed,passed


,timestamp,impression_uid,user_code,history_length,retained_length,valid_interests,has_text,has_entity,has_recent
0,2019-11-09 00:00:00,472831,337187,37,37,3,True,True,False
1,2019-11-09 00:00:02,526311,436772,78,50,3,True,True,False
2,2019-11-09 00:00:03,709414,80730,12,12,3,True,True,False


,split,representation,cohort,denominator,valid,fallback,valid_fraction,stage
0,test,entity,articles_unexposed_in_train,9819542,8269243,1550299,0.842121,prequential-all
1,test,entity,clicked_exposures,574845,495300,79545,0.861624,prequential-all
2,test,entity,new_users_vs_train,2306476,1820394,486082,0.789253,prequential-all
3,test,entity,no_causal_history,47385,37457,9928,0.790482,prequential-all
4,test,entity,returning_users_vs_train,11779081,9294705,2484376,0.789086,prequential-all
...,...,...,...,...,...,...,...,...
86,test,article_entity,split_catalogue,72023,62769,9254,0.871513,entity
87,all,supplied_text_profile,all_users,750434,736348,14086,0.981230,supplied_profiles
88,train,supplied_text_profile,training_users,667880,656044,11836,0.982278,supplied_profiles
89,all,supplied_entity_profile,all_users,750434,732743,17691,0.976426,supplied_profiles


,stage,wall_seconds,sampled_peak_process_rss_mb,platform,cpu,logical_cpus,ram_gb
19,text,6.703648,1223.868416,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869
20,entity,3.110573,1042.087936,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869
21,representation_coverage_audit,7.528604,1749.123072,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869
22,availability,0.844262,1912.291328,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869
23,supplied_profiles,26.376266,3736.535040,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869


## Handoff

The shared `TemporalState` class exposes causal text/entity/recent profiles, category counts, interest vectors,
and padded history-row sequences/masks. The feature build writes per-request history references so
retrieval, attention rankers and future ablations can reconstruct the same profile. State is timestamp-scoped;
a final user vector must never be reused for an earlier request. The disk pair store is replay scratch space,
not a resumable checkpoint. Persisted resume support is not implemented in this version.